# Processed Plume-Cell Run Comparison Diagnostics

Standalone notebook for balanced comparison figures using condensed `data/processed` plume-cell files:
1. Joint frequency `qfw` vs `qw`
2. Diameter-resolved histogram summary (`qfw`/`qw`)
3. Threshold occurrence/exceedance (`qfw`, `qw`) with run-difference panels

## Compute Playbook for Very Large NetCDF (1.6 TB class)

Use these rules by default for diagnostics notebooks:

- Avoid full `persist()` on high-dimensional datasets (`time,z,y,x[,bin]`).
- Avoid `.values` on large lazy arrays unless aggressively downsampled first.
- Convert heavy science fields to `float32` for diagnostics (`.astype('float32')`).
- Derive histogram bins from sampled subsets (stride in `time/altitude/lat/lon/diameter`).
- Compute expensive histograms on sampled data, not full-resolution 5D volumes.
- Keep `.compute()` targeted to compact intermediates (histograms, 2D/3D summaries).

Why this matters:

- A single `time,z,y,x,bin` variable at your dimensions is roughly **300+ GiB** in `float64`.
- Two runs + temporary arrays can exceed node RAM and crash the kernel.
- Stride-based summaries are usually stable enough for figure diagnostics at a fraction of the cost.

Recommended execution order:

1. Restart kernel before heavy runs.
2. Run cells sequentially (do not launch overlapping heavy cells).
3. If runtime/memory is high, increase stride before requesting more resources.
4. Only move to larger allocations when full-resolution science output is required.

In [ ]:
import os
import glob
import json
import re
from pathlib import Path
from time import perf_counter

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from xhistogram.xarray import histogram as xhist
import matplotlib as mpl
from tqdm.auto import tqdm

from dask.diagnostics.progress import ProgressBar

from utilities import define_bin_boundaries, load_plume_path_runs





BASE_STYLE = {
    "font.size": 15.5,
    "font.weight": "normal",
    "axes.titlepad": 5.0,
    "xtick.top": True,
    "xtick.major.top": True,
    "xtick.major.width": 1.0,
    "xtick.major.size": 3.0,
    "ytick.major.width": 1.0,
    "ytick.major.size": 3.0,
    "ytick.minor.width": 1.0,
    "ytick.minor.size": 3.0,
    "ytick.right": True,
    "axes.linewidth": 1.0,
    "savefig.dpi": 300,
    "figure.dpi": 110,
}


STYLE_TIMESERIES = {
    **BASE_STYLE,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "lines.linewidth": 1.8,
    "lines.markersize": 4.0,
    "legend.frameon": False,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
}


STYLE_2D = {
    **BASE_STYLE,
    "axes.grid": False,
    "image.interpolation": "nearest",
    "xtick.minor.visible": False,
    "ytick.minor.visible": False,
    "axes.labelpad": 4.0,
}


STYLE_HIST = {
    **BASE_STYLE,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "grid.linewidth": 0.5,
    "patch.edgecolor": "black",
    "patch.linewidth": 0.7,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
}


STYLE_REGISTRY = {
    "timeseries": STYLE_TIMESERIES,
    "2d": STYLE_2D,
    "hist": STYLE_HIST,
}


def get_style(kind: str) -> dict:
    try:
        return STYLE_REGISTRY[kind]
    except KeyError as exc:
        valid = ", ".join(STYLE_REGISTRY.keys())
        raise ValueError(f"Unknown style '{kind}'. Use one of: {valid}") from exc


def use_style(kind: str):
    return mpl.rc_context(get_style(kind))



xr.set_options(keep_attrs=True)

model_data_root = '/work/bb1262/user/schimmel/cosmo-specs-torch/cosmo-specs-runs/'
cs_runs = [
    # ['cs-eriswil__20260123_180947', '200x160', 1],
    # ['cs-eriswil__20260123_180947', '200x160', 2],
    #{"label": "400m, inp 1e6, ccn 400 (columnar)", "cs_run": 
    ["cs-eriswil__20251125_114053", '50x40', 1],
    ["cs-eriswil__20251125_114053", '50x40', 0],
    # ["cs-eriswil__20260127_211338", "20260127211431"],
    # ["cs-eriswil__20260127_211338", "20260127211551"],
]
tracking_by_var = 'qi+qs'

In [20]:
def _extract_exp_ids(cs_run, processed_root):
    patt = re.compile(rf"^data_{re.escape(cs_run)}_(\d{{14}})_integrated_plume_path_.*_cell\d+\.nc$")
    exp_ids = set()
    for p in Path(processed_root).glob(f"data_{cs_run}_*_integrated_plume_path_*_cell*.nc"):
        m = patt.match(p.name)
        if m:
            exp_ids.add(m.group(1))
    return sorted(exp_ids)


def _load_run_mod(cs_run, ep_dom, cs_run_idx, tracking_by_var='qi+qs'):
    processed_root = Path('../../data/processed')
    exp_ids = _extract_exp_ids(cs_run, processed_root)
    if not exp_ids:
        raise FileNotFoundError(f'No processed plume-path files found for run {cs_run} in {processed_root}')
    if cs_run_idx >= len(exp_ids):
        raise IndexError(f'cs_run_idx={cs_run_idx} out of range for {cs_run} with exp ids: {exp_ids}')

    exp_id = exp_ids[cs_run_idx]
    runs = [{'label': f'{cs_run}:{exp_id}', 'cs_run': cs_run, 'exp_id': exp_id}]
    datasets = load_plume_path_runs(runs=runs, processed_root=processed_root, kinds=('integrated',))
    ds = datasets[runs[0]['label']]['integrated']

    rename_dims = {}
    if 'bins' in ds.dims:
        rename_dims['bins'] = 'diameter'
    if rename_dims:
        ds = ds.rename(rename_dims)

    required = ['qfw', 'qw', 'nf', 'nw']
    missing = [v for v in required if v not in ds.data_vars]
    if missing:
        raise KeyError(f'Missing required processed variables: {missing}; available={list(ds.data_vars)}')

    mod = ds[required].astype('float32')
    if 'diameter' not in mod.coords:
        mod = mod.assign_coords({'diameter': xr.DataArray(np.arange(mod.sizes['diameter']), dims='diameter')})

    d_um = define_bin_boundaries() * 1.0e6 * 2.0
    if d_um.size == mod.sizes['diameter'] + 1:
        mod = mod.assign_coords({'diameter_edges': xr.DataArray(d_um, dims='diameter_edges')})

    mod[tracking_by_var] = xr.where(mod['qfw'] + mod['qw'] > 0, mod['qfw'] + mod['qw'], np.nan)
    mod[tracking_by_var].attrs['units'] = mod['qfw'].attrs.get('units', '')
    return mod, exp_id


def _sample_for_bins(da, stride=None):
    stride = stride or {'time': 16, 'diameter': 4, 'cell': 2}
    sel = {dim: slice(None, None, stride[dim]) for dim in da.dims if dim in stride}
    return da.isel(sel)


def _safe_log_bins(da, nbins=72, stride=None, lower_pct=0.2, upper_pct=99.8):
    sample = _sample_for_bins(da, stride=stride)
    values = np.asarray(sample.where(np.isfinite(sample) & (sample > 0)).compute().values).ravel()
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.logspace(-12, -6, nbins)
    vmin, vmax = np.nanpercentile(values, [lower_pct, upper_pct])
    vmin = max(float(vmin), np.finfo(float).tiny)
    vmax = max(float(vmax), vmin * 10.0)
    return np.logspace(np.log10(vmin), np.log10(vmax), nbins)


def _robust_bounds_linear(da, lower_pct=0.2, upper_pct=99.8):
    values = np.asarray(da.where(np.isfinite(da)).values).ravel()
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0, 1.0
    lo, hi = np.nanpercentile(values, [lower_pct, upper_pct])
    if lo == hi:
        hi = lo + 1e-12
    return float(lo), float(hi)


def _compute_joint_freq(mod, xbins, ybins, x_var='qfw', y_var='qw'):
    h = xhist(mod[x_var], mod[y_var], bins=[xbins, ybins]).compute()
    return h / h.sum()


def _compute_diameter_resolved_qfreq(mod, xbins, ybins, stride=None, show_progress=True):
    stride = stride or {'time': 2, 'cell': 2, 'diameter': 2}
    sel = {
        'time': slice(None, None, stride['time']),
        'diameter': slice(None, None, stride['diameter']),
    }
    if 'cell' in mod.dims:
        sel['cell'] = slice(None, None, stride['cell'])
    sampled = mod.isel(sel)
    rows = []
    iterator = tqdm(range(sampled.diameter.size), desc='diameter bins', leave=True) if show_progress else range(sampled.diameter.size)
    for ibin in iterator:
        mo = sampled.isel(diameter=ibin)
        h = xhist(mo['qfw'], mo['qw'], bins=[xbins, ybins]).sum('qw_bin').compute().expand_dims('diameter')
        rows.append(h)
    ds = xr.concat(rows, dim='diameter').assign_coords({'diameter': sampled.diameter})
    return ds / ds.sum('qfw_bin')


def _compute_exceedance(mod, var, threshold):
    da = xr.where(mod[var] >= threshold, 1.0, 0.0)
    reduce_dims = [d for d in da.dims if d not in ('time', 'diameter')]
    if reduce_dims:
        da = da.mean(dim=reduce_dims)
    return da.compute()

In [21]:
PRESETS = {
    'fast': {
        'nbins': 32,
        'bin_stride': {'time': 24, 'diameter': 4, 'cell': 2},
        'diam_stride': {'time': 8, 'diameter': 4, 'cell': 2},
    },
    'balanced': {
        'nbins': 64,
        'bin_stride': {'time': 16, 'diameter': 2, 'cell': 1},
        'diam_stride': {'time': 4, 'diameter': 2, 'cell': 1},
    },
}

preset = 'fast'  # switch to 'balanced' for higher detail
run_cfg = PRESETS[preset]
q_low, q_high = 0.2, 99.8  # central 99.6% interval
print(f"[cfg] preset={preset}  nbins={run_cfg['nbins']}  diam_stride={run_cfg['diam_stride']}")
print(f"[cfg] robust bounds percentile window: {q_low:.1f}%..{q_high:.1f}%")

start_all = perf_counter()

def _log_stage(label, t0):
    dt = perf_counter() - t0
    print(f'[stage] {label:<30} {dt:7.1f} s')
    return perf_counter()

print('[info] Starting comparison workflow...')
t0 = perf_counter()
run_data = []
for cs_run, ep_dom, cs_run_idx in tqdm(cs_runs, desc='load runs'):
    mod_i, flare_name_i = _load_run_mod(cs_run, ep_dom, cs_run_idx, tracking_by_var=tracking_by_var)
    run_data.append({'run_id': cs_run, 'flare_exp_name': flare_name_i, 'mod': mod_i})
t0 = _log_stage('load runs', t0)

mod1, mod2 = run_data[0]['mod'], run_data[1]['mod']
label1, label2 = run_data[0]['flare_exp_name'], run_data[1]['flare_exp_name']

joint_x_var, joint_y_var = 'qfw', 'qw'
qv_bins  = _safe_log_bins(xr.concat([mod1[joint_x_var], mod2[joint_x_var]], dim='stack_x', join='outer'), nbins=run_cfg['nbins'], stride=run_cfg['bin_stride'], lower_pct=q_low, upper_pct=q_high)
qc_bins  = _safe_log_bins(xr.concat([mod1[joint_y_var], mod2[joint_y_var]], dim='stack_y', join='outer'), nbins=run_cfg['nbins'], stride=run_cfg['bin_stride'], lower_pct=q_low, upper_pct=q_high)
qfw_bins = _safe_log_bins(xr.concat([mod1['qfw'], mod2['qfw']], dim='stack_qfw', join='outer'), nbins=run_cfg['nbins'], stride=run_cfg['bin_stride'], lower_pct=q_low, upper_pct=q_high)
qw_bins  = _safe_log_bins(xr.concat([mod1['qw'],  mod2['qw']],  dim='stack_qw', join='outer'),  nbins=run_cfg['nbins'], stride=run_cfg['bin_stride'], lower_pct=q_low, upper_pct=q_high)
t0 = _log_stage('derive bins', t0)

qv_xlim = (float(qv_bins[0]), float(qv_bins[-1]))
qc_ylim = (float(qc_bins[0]), float(qc_bins[-1]))
qfw_ylim = (float(qfw_bins[0]), float(qfw_bins[-1]))
print(f"[cfg] {joint_x_var} x-limits: {qv_xlim[0]:.2e} .. {qv_xlim[1]:.2e}")
print(f"[cfg] {joint_y_var} y-limits: {qc_ylim[0]:.2e} .. {qc_ylim[1]:.2e}")
print(f"[cfg] qfw y-limits: {qfw_ylim[0]:.2e} .. {qfw_ylim[1]:.2e}")

joint1 = _compute_joint_freq(mod1, qv_bins, qc_bins, x_var=joint_x_var, y_var=joint_y_var)
joint2 = _compute_joint_freq(mod2, qv_bins, qc_bins, x_var=joint_x_var, y_var=joint_y_var)
joint_diff = joint2 - joint1
t0 = _log_stage('joint histogram compute', t0)

# Figure A: Joint qfw/qw frequency
with use_style('hist'):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
    vmax_joint = float(np.nanmax([joint1.max().values, joint2.max().values]))
    vmin_joint = max(vmax_joint * 1e-5, np.finfo(float).tiny)

    for i, (da, ttl) in enumerate([(joint1, label1), (joint2, label2)]):
        xr.where(da > 0, da, np.nan).plot(
            ax=ax[i],
            xscale='log',
            yscale='log',
            cmap='viridis',
            norm=LogNorm(vmin=vmin_joint, vmax=vmax_joint),
            cbar_kwargs={'label': 'joint frequency'},
        )
        ax[i].set_title(f'Joint freq {joint_x_var}/{joint_y_var}: {ttl}')
        ax[i].set_xlabel(joint_x_var)
        ax[i].set_ylabel(joint_y_var)
        ax[i].set_xlim(*qv_xlim)
        ax[i].set_ylim(*qc_ylim)

    vd = float(np.nanmax(np.abs(joint_diff.values)))
    if vd == 0:
        vd = 1e-12
    joint_diff.plot(
        ax=ax[2],
        xscale='log',
        yscale='log',
        cmap='coolwarm',
        vmin=-vd,
        vmax=vd,
        cbar_kwargs={'label': f'freq diff ({label2} - {label1})'},
    )
    ax[2].set_title(f'Joint freq difference ({joint_x_var}/{joint_y_var})')
    ax[2].set_xlabel(joint_x_var)
    ax[2].set_ylabel(joint_y_var)
    ax[2].set_xlim(*qv_xlim)
    ax[2].set_ylim(*qc_ylim)
    fig.savefig('fig-comparison-a-joint-freq-200x160.png', dpi=300)
t0 = _log_stage('figure A saved', t0)

# Figure B: Diameter-resolved qfw/qw summary
print('[info] Computing diameter-resolved histograms...')
diam_q1 = _compute_diameter_resolved_qfreq(mod1, qfw_bins, qw_bins, stride=run_cfg['diam_stride'], show_progress=True)
diam_q2 = _compute_diameter_resolved_qfreq(mod2, qfw_bins, qw_bins, stride=run_cfg['diam_stride'], show_progress=True)
diam_qdiff = diam_q2 - diam_q1
qdiam_xlim = (float(diam_q1.diameter.min().values), float(diam_q1.diameter.max().values))
t0 = _log_stage('diameter histogram compute', t0)

with use_style('hist'):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
    vmax_d = float(np.nanmax([diam_q1.max().values, diam_q2.max().values]))
    vmin_d = max(vmax_d * 1e-5, np.finfo(float).tiny)

    for i, (da, ttl) in enumerate([(diam_q1, label1), (diam_q2, label2)]):
        xr.where(da > 0, da, np.nan).plot(
            ax=ax[i],
            x='diameter',
            y='qfw_bin',
            xscale='log',
            yscale='log',
            cmap='magma',
            norm=LogNorm(vmin=vmin_d, vmax=vmax_d),
            cbar_kwargs={'label': 'freq(qfw | diameter)'},
        )
        ax[i].set_title(f'Diameter-resolved qfw/qw: {ttl}')
        ax[i].set_xlabel('diameter (um)')
        ax[i].set_ylabel('qfw')
        ax[i].set_xlim(*qdiam_xlim)
        ax[i].set_ylim(*qfw_ylim)

    vdd = float(np.nanmax(np.abs(diam_qdiff.values)))
    if vdd == 0:
        vdd = 1e-12
    diam_qdiff.plot(
        ax=ax[2],
        x='diameter',
        y='qfw_bin',
        xscale='log',
        yscale='log',
        cmap='coolwarm',
        vmin=-vdd,
        vmax=vdd,
        cbar_kwargs={'label': f'freq diff ({label2} - {label1})'},
    )
    ax[2].set_title('Diameter-resolved difference')
    ax[2].set_xlabel('diameter (um)')
    ax[2].set_ylabel('qfw')
    ax[2].set_xlim(*qdiam_xlim)
    ax[2].set_ylim(*qfw_ylim)
    fig.savefig('fig-comparison-b-diameter-resolved-hist-200x160.png', dpi=300)
t0 = _log_stage('figure B saved', t0)

# Figure C: Threshold occurrence time-diameter
thresholds = {}
for var in ('qfw', 'qw'):
    sample = xr.concat([_sample_for_bins(mod1[var]), _sample_for_bins(mod2[var])], dim='sample_var')
    vals = np.asarray(sample.where(np.isfinite(sample) & (sample > 0)).compute().values).ravel()
    vals = vals[np.isfinite(vals)]
    thresholds[var] = float(np.nanpercentile(vals, 90)) if vals.size else 1e-12

occ = {}
for var, th in tqdm(thresholds.items(), desc='threshold occurrence'):
    occ[(label1, var)] = _compute_exceedance(mod1, var, th)
    occ[(label2, var)] = _compute_exceedance(mod2, var, th)
t0 = _log_stage('threshold occurrence compute', t0)

with use_style('2d'):
    fig, ax = plt.subplots(2, 3, figsize=(15, 7.2), constrained_layout=True, sharex=True, sharey=True)
    for r, var in enumerate(['qfw', 'qw']):
        da1 = occ[(label1, var)]
        da2 = occ[(label2, var)]
        dad = da2 - da1

        occ_lo, occ_hi = _robust_bounds_linear(xr.concat([da1, da2], dim='run'), lower_pct=q_low, upper_pct=q_high)
        occ_lo = max(0.0, occ_lo)
        occ_hi = min(1.0, occ_hi)
        if occ_lo >= occ_hi:
            occ_lo, occ_hi = 0.0, 1.0

        d_lo, d_hi = _robust_bounds_linear(dad, lower_pct=q_low, upper_pct=q_high)
        vm = max(abs(d_lo), abs(d_hi), 1e-12)

        for c, (da, ttl, cmap) in enumerate([
            (da1, f'{label1}: {var} >= {thresholds[var]:.1e}', 'viridis'),
            (da2, f'{label2}: {var} >= {thresholds[var]:.1e}', 'viridis'),
            (dad, f'diff ({label2} - {label1})', 'coolwarm'),
        ]):
            kw = dict(x='time', y='diameter', cmap=cmap, add_colorbar=True)
            if c < 2:
                kw['vmin'] = occ_lo
                kw['vmax'] = occ_hi
                kw['cbar_kwargs'] = {'label': 'occurrence frequency'}
            else:
                kw['vmin'] = -vm
                kw['vmax'] = vm
                kw['cbar_kwargs'] = {'label': 'frequency difference'}
            da.plot(ax=ax[r, c], **kw)
            ax[r, c].set_title(ttl)
            ax[r, c].set_xlabel('time')
            ax[r, c].set_ylabel('diameter')

    fig.savefig('fig-comparison-c-threshold-occurrence-200x160.png', dpi=300)
t0 = _log_stage('figure C saved', t0)

print('Saved: fig-comparison-a-joint-freq-200x160.png')
print('Saved: fig-comparison-b-diameter-resolved-hist-200x160.png')
print('Saved: fig-comparison-c-threshold-occurrence-200x160.png')
print(f'[done] total elapsed {perf_counter() - start_all:7.1f} s')

[cfg] preset=fast  nbins=32  diam_stride={'time': 8, 'altitude': 2, 'latitude': 4, 'longitude': 4, 'diameter': 6}
[cfg] robust bounds percentile window: 0.2%..99.8%
[info] Starting comparison workflow...


load runs:   0%|          | 0/2 [00:00<?, ?it/s]

[stage] load runs                          1.3 s


/tmp/ipykernel_3438824/2397891858.py:38: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  qv_bins  = _safe_log_bins(xr.concat([mod1['qv'],  mod2['qv']],  dim='stack_qv'),  nbins=run_cfg['nbins'], stride=run_cfg['bin_stride'], lower_pct=q_low, upper_pct=q_high)
/tmp/ipykernel_3438824/2397891858.py:39: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  qc_bins  = _saf

[stage] derive bins                        7.1 s
[cfg] qv x-limits: 2.11e+00 .. 2.11e+01
[cfg] qc y-limits: 6.88e-06 .. 4.15e-01
[cfg] qfw y-limits: 4.51e-19 .. 4.46e-05
[stage] joint histogram compute           54.2 s
[stage] figure A saved                     1.5 s
[info] Computing diameter-resolved histograms...


diameter bins:   0%|          | 0/11 [00:00<?, ?it/s]

diameter bins:   0%|          | 0/11 [00:00<?, ?it/s]

[stage] diameter histogram compute        83.5 s
[stage] figure B saved                     1.0 s


threshold occurrence:   0%|          | 0/2 [00:00<?, ?it/s]

[stage] threshold occurrence compute     103.3 s
[stage] figure C saved                     1.4 s
Saved: fig-comparison-a-joint-freq-200x160.png
Saved: fig-comparison-b-diameter-resolved-hist-200x160.png
Saved: fig-comparison-c-threshold-occurrence-200x160.png
[done] total elapsed   253.2 s
